# Fairness-Aware GPT-2 — QQP Batch Run

**Built for `Save Version → Save & Run All (Commit)`.** Runs unattended in a
fresh container. Close your laptop; it keeps going.

Trains one model (`cda_reg`, 5 epochs) and writes the Table 2 numbers.

---

## Before you commit

Right panel → **Session options**:

| Setting | Value |
|---|---|
| **Accelerator** | **GPU T4 x2** — NOT P100 (Pascal sm_60 is unsupported by current PyTorch) |
| **Internet** | **On** |

Then top right → **Save Version** → **Save & Run All (Commit)** → **Save**.

Close the tab. Watch the **Versions** tab; results land in that version's **Output**.

## Why a separate notebook

"Run All" means *all* cells. The interactive notebook has optional `baseline` and
`cda` runs and a Hugging Face login at the end — unattended, those would burn
hours you didn't intend or fail on a missing token. This does exactly one job.

## 1. Config — edit `REPO`, then commit

In [ ]:
import os

REPO = "https://github.com/Ricky11234/fairness-aware-gpt2"
MODE = "cda_reg"  # cda_reg | cda | baseline
EPOCHS = 5  # Table 4 reports 5-epoch numbers, so this is comparable
TRAIN_SIZE = 283011  # matches the report's pair count (GLUE ships 363,846)
LAMBDA_FAIR = 0.5

# Exported now, before any shell cell needs them.
os.environ.update(
    MODE=MODE,
    EPOCHS=str(EPOCHS),
    TRAIN_SIZE=str(TRAIN_SIZE),
    LAMBDA_FAIR=str(LAMBDA_FAIR),
)
print(f"{MODE} | {EPOCHS} epochs | {TRAIN_SIZE:,} train pairs")

## 2. Fail fast

Each check aborts rather than letting a 2-hour job discover the problem at the
end. Nobody is watching a batch run, so an early hard failure beats a late soft one.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU T4 x2 and re-commit."
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f"GPU: {name} | compute capability {cap[0]}.{cap[1]}")
assert cap >= (7, 0), (
    f"{name} is compute {cap[0]}.{cap[1]}; this PyTorch needs >= 7.0. "
    "P100 (6.0) has no compiled kernels — switch the Accelerator to GPU T4 x2."
)
print("GPU OK")

## 3. Clone

In [ ]:
%cd /kaggle/working
!rm -rf fairness-repo
!git clone -q $REPO fairness-repo
%cd fairness-repo

In [ ]:
# identity.py both generates the CDA counterfactuals AND computes the flip rate,
# so stale code silently corrupts the headline number. Fail now, not in 2 hours.
from pathlib import Path

assert "AMBIGUOUS_PRONOUNS" in Path("src/fairness_gpt2/identity.py").read_text(), (
    "Cloned repo is missing the pronoun fix — you have unpushed changes. "
    "GitHub Desktop > Push origin, then re-commit."
)
print("Repo is current")

In [ ]:
!pip install -q "transformers>=4.41" "datasets>=2.19"

## 4. Data

In [ ]:
!PYTHONPATH=src python scripts/download_data.py --only qqp --train-size $TRAIN_SIZE --skip-test

In [ ]:
import sys

sys.path.insert(0, "src")  # must come before importing the package
from fairness_gpt2.data import load_qqp

n_train, n_dev = len(load_qqp("data/quora-train.csv")), len(load_qqp("data/quora-dev.csv"))
print(f"train: {n_train:,}  dev: {n_dev:,}")
assert n_dev == 40430, f"dev is {n_dev}, expected 40430 — metrics won't be comparable"
print("Data OK")

## 5. Smoke test

Two minutes. Cheap insurance: if the pipeline is broken you find out at minute 3
instead of losing the session.

In [ ]:
!PYTHONPATH=src python -m fairness_gpt2.train --mode $MODE --train data/quora-train.csv --dev data/quora-dev.csv --out /tmp/smoke --epochs 1 --train-subset 2000 --eval-subset 1000

## 6. The real run

~2 hours on T4. `--save-half` writes fp16 (~250MB) so the checkpoint fits
Streamlit Cloud's memory limit.

Expect a lot of progress-bar lines — normal for batch.

In [ ]:
!PYTHONPATH=src python -m fairness_gpt2.train --mode $MODE --train data/quora-train.csv --dev data/quora-dev.csv --out /kaggle/working/checkpoints/$MODE --epochs $EPOCHS --lambda-fair $LAMBDA_FAIR --save-half

## 7. Collect outputs

Everything under `/kaggle/working` is saved with the version. Copying the small
artifacts to the top level makes them easy to find in the Output tab.

In [ ]:
import json
import shutil

shutil.copytree("results/reproduced", "/kaggle/working/results/reproduced", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/reproduced", "zip", "results", "reproduced")

metrics = json.loads(Path(f"results/reproduced/{MODE}.json").read_text())
print(json.dumps(metrics, indent=2))

In [ ]:
# Side by side with the report. Table 4 gives 5-epoch numbers for the CDA runs;
# the baseline row is 10-epoch, so that comparison is indicative only.
REPORTED = {
    "cda": {"dev_acc": 0.8835, "subgroup_gap": 0.0433, "flip_rate": 0.0392},
    "cda_reg": {"dev_acc": 0.8856, "subgroup_gap": 0.0512, "flip_rate": 0.0296},
    "baseline": {"dev_acc": 0.8955, "subgroup_gap": 0.0365, "flip_rate": 0.0456},
}
r = REPORTED.get(MODE, {})
print(f"{'metric':<15}{'paper':>10}{'yours':>10}{'delta':>10}")
print("-" * 45)
for key, mine in (
    ("dev_acc", metrics["accuracy"]),
    ("subgroup_gap", metrics["subgroup_gap"]),
    ("flip_rate", metrics["flip_rate"]),
):
    p = r.get(key)
    print(
        f"{key:<15}{p if p is not None else '-':>10}{mine:>10.4f}"
        f"{(f'{mine - p:+.4f}' if p is not None else '-'):>10}"
    )

print(f"\nflips: {metrics['flips']} of {metrics['n_identity']} identity-bearing dev examples")
print("\nFlip rate is EXPECTED to differ. The report gives substitution counts")
print("(60/20/22) and three examples, not the lists — identity.py reconstructs")
print("them, so the counterfactuals aren't identical. Accuracy should land close.")

## When it finishes

1. **Versions** tab → your run → **Output**
2. Download `reproduced.zip` → unzip into your project's `results\reproduced\`
3. Commit + push → your live app shows your numbers beside the paper's
4. Download `checkpoints/cda_reg/` if you want it on Hugging Face

The HF upload is deliberately left out — do that interactively so your token
stays out of an unattended run.

## Notes

- Kaggle gives **30 GPU-hours/week**; sessions cap at 12h. One 5-epoch run is
  ~2h, so there's room.
- For the other two models: change `MODE` and commit again.